# 13 · Sub-district boundaries (India) — geoBoundaries ADM3

**Source:** geoBoundaries open release, India ADM3 (sub-districts / tehsils / taluks), boundary year 2018, from
Pathways Data Pvt. Ltd. / lgdirectory.gov.in. **Licence:** ODbL 1.0. Cite: Runfola, D. et al. (2020)
*PLoS ONE* 15(4): e0231866.

Used to summarise the all-India results below district level. (Kerala local bodies come from OSM, notebook 14.)

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import json
import shutil
import time

import requests

OUT = PROJECT / "data" / "raw" / "boundaries"
API = "https://www.geoboundaries.org/api/current/gbOpen/IND/ADM3/"
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

### `download()`

In [3]:
def download(url, f, min_free_gb=5):
    """Download once. A finished file keeps its name; an interrupted one stays *.part and is fetched again."""
    f.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(4):  # servers can be slow to answer; wait and retry
        try:
            size = int(requests.head(url, headers=HEADERS, allow_redirects=True, timeout=300).headers.get("content-length", 0))
            break
        except requests.RequestException as e:
            print("  server slow, retrying:", type(e).__name__)
            time.sleep(60)
    else:
        raise SystemExit(f"no answer from {url}")
    if f.exists() and (size == 0 or f.stat().st_size == size):
        print(f"already saved: {f.name} ({f.stat().st_size / 1e6:.0f} MB)")
        return f
    free = shutil.disk_usage(f.parent).free
    if free - size < min_free_gb * 1e9:
        raise SystemExit(f"not enough disk for {f.name}: {size / 1e9:.2f} GB needed, {free / 1e9:.1f} GB free")
    part = f.with_name(f.name + ".part")
    with requests.get(url, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(part, "wb") as out:
            shutil.copyfileobj(r.raw, out, length=1 << 20)
    if size and part.stat().st_size != size:
        raise SystemExit(f"{f.name}: got {part.stat().st_size} bytes, expected {size}")
    part.replace(f)
    print(f"downloaded: {f.name} ({size / 1e6:.0f} MB)")
    return f

## Run

In [4]:
info = requests.get(API, timeout=60).json()
print("ADM3 | year:", info["boundaryYearRepresented"], "| units:", info["admUnitCount"], "| licence:",
      info["boundaryLicense"], "| source:", info["boundarySource"], "| built:", info["buildDate"])
gj = info["gjDownloadURL"]
download(gj, OUT / "geoBoundaries-IND-ADM3.geojson")
download(gj.replace(".geojson", "-metaData.txt"), OUT / "geoBoundaries-IND-ADM3-metaData.txt")

ADM3 | year: 2018 | units: 6836 | licence: Open Data Commons Open Database License 1.0 | source: Pathways Data Pvt. Ltd., lgdirectory.gov.in | built: Dec 12, 2023


downloaded: geoBoundaries-IND-ADM3.geojson (106 MB)


downloaded: geoBoundaries-IND-ADM3-metaData.txt (0 MB)


WindowsPath('C:/Users/ASUS/Desktop/sem3/BIG SPATIAL DATA/PROJECT- SPATIAL INEQUALITY ASESSMENT/data/raw/boundaries/geoBoundaries-IND-ADM3-metaData.txt')

## Check

In [5]:
d = json.loads((OUT / "geoBoundaries-IND-ADM3.geojson").read_text(encoding="utf-8"))["features"]
print(len(d), "sub-district features in file | metadata admUnitCount", info["admUnitCount"])
print("example:", d[0]["properties"])

6824 sub-district features in file | metadata admUnitCount 6836
example: {'shapeName': 'Ghograpar (Pt)', 'shapeISO': '', 'shapeID': '7132399B28259355418462', 'shapeGroup': 'IND', 'shapeType': 'ADM3'}
